In [ ]:
from orion.algos.human_video_oogs import HumanVideoOOGs
from orion.algos.oog import OpenWorldObjectSceneGraph
from orion.utils.misc_utils import *
from orion.utils.o3d_utils import *
from orion.utils.correspondence_utils import CorrespondenceModel
import h5py
import numpy as np
human_video_oogs = HumanVideoOOGs()
# correspondence_model = CorrespondenceModel()
# human_video_oogs.set_correspondence_model(correspondence_model)

In [ ]:
human_video_annotation_path = "annotations/human_demo/iphone_front_boat_demo"
masks = np.load(os.path.join(human_video_annotation_path, "masks.npz"))["arr_0"]

#rescaling zoe depth estimation back to original
demo_file = h5py.File("/home/arisrei/research/RPL/ORION/datasets/iphones/iphone_front_boat/iphone_front_boat_demo.hdf5", "r+")
out_uint16 = demo_file.get("data/human_demo/obs/zoe_depth")[()]
max_and_mins = demo_file.get("data/human_demo/obs/rescale_max_min")[()]
old_max = max_and_mins[:,0]
old_min = max_and_mins[:,1]

dif = (old_max-old_min).reshape(out_uint16.shape[0],1,1,1)
old_min_reshaped = old_min.reshape(out_uint16.shape[0],1,1,1)

zoe_depth = (((out_uint16 / 65535.0) * dif) + old_min_reshaped).astype(np.float32)
zoe_depth = zoe_depth.reshape((zoe_depth.shape[0], zoe_depth.shape[1], zoe_depth.shape[2])) #squeeze the last dimension

avg_dif = np.mean(zoe_depth - zoe_depth[0],axis=(1,2))
print(avg_dif.shape, avg_dif)

mult=-1

new_depth = zoe_depth - (mult*avg_dif.reshape(avg_dif.shape[0],1,1))

if "data/human_demo/obs/tmp_agentview_depth" in demo_file:
    del demo_file["data/human_demo/obs/tmp_agentview_depth"]

tmp_agentview_depth = demo_file.get("data/human_demo/obs/agentview_depth")[()]
if "data/human_demo/obs/agentview_depth" in demo_file:
    del demo_file["data/human_demo/obs/agentview_depth"]

demo_file.create_dataset("data/human_demo/obs/tmp_agentview_depth", data=tmp_agentview_depth)
demo_file.create_dataset("data/human_demo/obs/agentview_depth", data=new_depth)

demo_file.close()
#new_depth.shape




In [ ]:
cnt = 0
dif  = 0
for i in range(zoe_depth[0].shape[0]):
    for j in range(zoe_depth[0].shape[1]):
        if masks[0][i][j] == 1:
            cnt += 1
            dif += zoe_depth[1][i][j] - zoe_depth[0][i][j] 

print(cnt)
dif / cnt
    

In [ ]:
human_video_oogs.init()
human_video_oogs.generate_from_human_video(human_video_annotation_path)

In [ ]:
demo_file = h5py.File("/home/arisrei/research/RPL/ORION/datasets/iphones/iphone_front_boat/iphone_front_boat_demo.hdf5", "r+")

#return agentview_depth back to original:

tmp_agentview_depth = demo_file.get("data/human_demo/obs/tmp_agentview_depth")[()]
if "data/human_demo/obs/tmp_agentview_depth" in demo_file:
    del demo_file["data/human_demo/obs/tmp_agentview_depth"]

if "data/human_demo/obs/agentview_depth" in demo_file:
    del demo_file["data/human_demo/obs/agentview_depth"]

demo_file.create_dataset("data/human_demo/obs/agentview_depth", data=tmp_agentview_depth)
demo_file.close()


In [ ]:
idx = 3

human_video_oogs.get_graph(0).draw_overlay_image(mode="object")

input_annotation = human_video_oogs.get_graph(idx).input_annotation
overlay_image = overlay_xmem_mask_on_image(
                human_video_oogs.get_graph(idx).input_image, 
                human_video_oogs.get_graph(idx).input_annotation,
                use_white_bg=False,
                rgb_alpha=0.3)



overlay_image = overlay_image * (input_annotation == 0)[..., None] + human_video_oogs.get_graph(idx).input_image * (input_annotation != 0)[..., None]
plotly_draw_image(overlay_image, width=640, height=480)


### Source image and depth

In [ ]:
plotly_draw_image(human_video_oogs.get_graph(2).input_image, width=640, height=480)
plotly_draw_image(human_video_oogs.get_graph(1).input_depth, width=640, height=480)
human_video_oogs.get_graph(0).draw_overlay_image(mode="point", width=640, height=480, default_point_color="rgb(30, 122, 122)")

### Draw the scene and the objects from human video
1. Draw the scene
2. Draw the scene with objects being annotated with segmentation mask
3. Trajectories that show the 3D motion of the objects

In [ ]:
human_video_oogs.get_graph(1).draw_scene_3d(downsample=True, 
                                            depth_trunc=3.0,  
                                            use_white_bg=True,
                                            no_background=True, 
                                            overlay_annotation=True)

In [ ]:
trajs = human_video_oogs.get_graph(1).get_world_trajs(object_ids=[2])
trajs = np.delete(trajs, 34, axis=0)
human_video_oogs.get_graph(1).draw_objects_list_3d(object_ids=[1, 2], downsample=False, no_background=True)

In [ ]:
trajs = human_video_oogs.get_graph(1).get_world_trajs(object_ids=[2])
trajs = np.delete(trajs, [10, 17], axis=0)
#trajs = trajs[0]
fig = human_video_oogs.get_graph(1).draw_scene_3d(
                                            additional_points=trajs,
                                            downsample=True, no_background=True,
                                            depth_trunc=2/ 1000,  overlay_annotation=True, 
                                            additional_point_draw_lines=True,offline=True)

fig.write_html("trajectory_pos_adjustment_all_avg.html")
fig.show()

### Draw individual object point clouds

In [ ]:
human_video_oogs.get_graph(1).draw_objects_3d(object_id=2, downsample=False,  no_background=True, draw_keypoints=True, keypoints_rgb_str="(30, 200, 200)", uniform_color=True)

In [ ]:
trajs = human_video_oogs.get_graph(1).get_world_trajs(object_ids=[1])
trajs = np.delete(trajs, [10, 17], axis=0)
human_video_oogs.get_graph(1).draw_objects_3d(object_id=1, downsample=False,  no_background=True, draw_trajs=False, additional_point_draw_lines=True, keypoints_rgb_str="(30, 122, 122)", 
                                              additional_points=trajs, 
                                              uniform_color=False)

In [ ]:

demo_file = h5py.File("/home/arisrei/research/RPL/ORION/datasets/iphones/iphone_front_boat/iphone_front_boat_demo.hdf5", "r+")
rgb = demo_file.get("data/human_demo/obs/agentview_rgb")[()]
out_uint16 = demo_file.get("data/human_demo/obs/zoe_depth")[()]
max_and_mins = demo_file.get("data/human_demo/obs/rescale_max_min")[()]
old_max = max_and_mins[:,0]
old_min = max_and_mins[:,1]

dif = (old_max-old_min).reshape(out_uint16.shape[0],1,1,1)
old_min_reshaped = old_min.reshape(out_uint16.shape[0],1,1,1)

rescale_back = (((out_uint16 / 65535.0) * dif) + old_min_reshaped).astype(np.float32)
zoe_depth = rescale_back.reshape((rescale_back.shape[0], rescale_back.shape[1], rescale_back.shape[2])) #squeeze the last dimension
data_config = json.loads(demo_file["data"].attrs["data_config"])
demo_file.close()


annotations_folder = "/home/arisrei/research/RPL/ORION/annotations/human_demo/iphone_front_boat_demo"
masks = np.load(os.path.join(annotations_folder, "masks.npz"))["arr_0"]

boat_mask = 1
red_obj_mask = 2

human_video_annotation_path = "annotations/human_demo/iphone_front_boat_demo"
masks = np.load(os.path.join(human_video_annotation_path, "masks.npz"))["arr_0"]

avg_dif = np.mean(zoe_depth - zoe_depth[0],axis=(1,2),where=(masks[0]==1))
# print(avg_dif.shape, avg_dif)

new_depth = zoe_depth - (avg_dif.reshape(avg_dif.shape[0],1,1))

camera_name = "camera_rs_0"

camera_intrinsics = data_config["intrinsics"][camera_name]
camera_intrinsics_matrix = get_intrinsics_matrix_from_dict(camera_intrinsics)
camera_extrinsics = data_config["extrinsics"][camera_name]
camera_extrinsics_matrix = get_extrinsics_matrix_from_dict(camera_extrinsics)

# print(zoe_depth.shape)
# zoe_depth = np.where(masks == 0, -10000, zoe_depth)
# print(zoe_depth.shape, np.unique(zoe_depth))

# d_1 = zoe_depth[1]
# d_2 = zoe_depth[60]


# rgb_1 = rgb[1]
# rgb_2 = rgb[60]

# pcd_points_1, pcd_colors_1 = scene_pcd_fn(
#     rgb_img_input=rgb_1,
#     depth_img_input=d_1,
#     extrinsic_matrix=camera_extrinsics_matrix,
#     intrinsic_matrix=camera_intrinsics_matrix,
#     downsample=False,
#     depth_trunc=3,
# )

# pcd_points_2, pcd_colors_2 = scene_pcd_fn(
#     rgb_img_input=rgb_2,
#     depth_img_input=d_2,
#     extrinsic_matrix=camera_extrinsics_matrix,
#     intrinsic_matrix=camera_intrinsics_matrix,
#     downsample=False,
#     depth_trunc=3,
# )

# pcd_points = np.concatenate([pcd_points_1, pcd_points_2], axis=0)
# pcd_colors = np.concatenate([pcd_colors_1, pcd_colors_2], axis=0)

# print(pcd_points.shape, pcd_colors.shape)
# print(np.unique(pcd_points[::2]))
# print(np.unique(d_1))

# marker_size = 3
# fig = plotly_draw_3d_pcd(pcd_points, pcd_colors, marker_size=marker_size, no_background=False, additional_point_draw_lines=False, offline=True)
# fig.write_html("two_frames_far_adjusted.html")
# fig.show()


In [ ]:
pcd_points_orig, pcd_colors_orig = scene_pcd_fn(
    rgb_img_input=rgb[0],
    depth_img_input=zoe_depth[0],
    extrinsic_matrix=camera_extrinsics_matrix,
    intrinsic_matrix=camera_intrinsics_matrix,
    downsample=True,
    depth_trunc=2,
    depth_scale=1
)

red_only = zoe_depth.copy()
red_only[masks != red_obj_mask] = 10000

skip = 5
for i in range(1, red_only.shape[0], skip):
    pcd_points, pcd_colors = scene_pcd_fn(
        rgb_img_input=rgb[i],
        depth_img_input=red_only[i],
        extrinsic_matrix=camera_extrinsics_matrix,
        intrinsic_matrix=camera_intrinsics_matrix,
        downsample=True,
        depth_trunc=2,
        depth_scale=1
    )
    pcd_points_orig = np.concatenate([pcd_points_orig, pcd_points], axis=0)
    pcd_colors_orig = np.concatenate([pcd_colors_orig, pcd_colors], axis=0)

marker_size = 3
fig = plotly_draw_3d_pcd(pcd_points_orig, pcd_colors_orig, marker_size=marker_size, no_background=False, additional_point_draw_lines=False, offline=True)
#fig.write_html("two_frames_far_adjusted.html")
fig.show()